# Kaggle Run From Git

Thin Kaggle notebook: pull the latest repo code, then either train or analyze.

**Set `MODE` in the config cell below.**

### `MODE = "analyze"` (normal — write up results from already-trained weights)
Attach the 4 model datasets + the image dataset, then run every cell top to
bottom: **config → pull code → analyze → curves → Grad-CAM → zip**.
The training cell skips itself, so a full run is safe.
Download `plant_research_bundle.zip` from the output panel when it finishes.

### `MODE = "train"`
Trains the models in `MODELS`, then the same analysis cells run on the result.
`SMOKE = True` trains **1 epoch** — a pipeline check only; those numbers must
never go into the report.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/Internship.git"
REPO_DIR = Path("/kaggle/working/Internship")
BRANCH = None

# --- What this run does ---------------------------------------------------
# "analyze" -> do NOT train. Stage the already-trained weights from the attached
#              Kaggle datasets and only run the analysis/curves/zip cells.
#              This is the normal mode for writing up results.
# "train"   -> train the models in MODELS from scratch, then analyze them.
MODE = "analyze"

# --- Run settings (only used when MODE == "train") ---
SMOKE = True            # True -> quick 1-epoch check; NEVER use its numbers in the report
EPOCHS = 50
BATCH_SIZE = 16
MODELS = ["custom_cnn_v2"]

# --- Tier-1 training options (defaults reproduce the old behaviour) ---
OPTIMIZER = "adamw"              # adam | adamw
LR_SCHEDULER = "cosine_warmup"   # none | cosine | cosine_warmup
LABEL_SMOOTHING = 0.1
LOSS = "cb_focal"                # ce | focal | class_balanced | cb_focal
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0

assert MODE in {"analyze", "train"}, f"MODE must be 'analyze' or 'train', got {MODE!r}"
print(f"MODE = {MODE}" + (f"  (SMOKE={SMOKE}, epochs={1 if SMOKE else EPOCHS})" if MODE == "train" else "  -> training cell will be skipped"))


In [ ]:
import subprocess

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "--all"], cwd=REPO_DIR, check=True)
    if BRANCH:
        subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
else:
    command = ["git", "clone"]
    if BRANCH:
        command.extend(["--branch", BRANCH])
    command.extend([REPO_URL, str(REPO_DIR)])
    subprocess.run(command, check=True)

print("Repository ready:", REPO_DIR)

In [ ]:
# --- Training (skipped unless MODE == "train") ---
# Guarded on purpose: running this in analyze mode would overwrite
# /kaggle/working/plant_training_outputs with a fresh (possibly 1-epoch smoke)
# model and make the analysis cell report those numbers instead of the real ones.
if MODE != "train":
    print(f'MODE = "{MODE}" -> skipping training. Run the analysis cell below.')
else:
    %cd /kaggle/working/Internship
    import subprocess

    epochs = 1 if SMOKE else EPOCHS
    if SMOKE:
        print("!! SMOKE=True -> 1 epoch only. These numbers are a pipeline check, NOT results.")
    cmd = [
        "python", "scripts/train_kaggle.py",
        "--epochs", str(epochs),
        "--batch-size", str(BATCH_SIZE),
        "--models", *MODELS,
        "--optimizer", OPTIMIZER,
        "--lr-scheduler", LR_SCHEDULER,
        "--label-smoothing", str(LABEL_SMOOTHING),
        "--loss", LOSS,
        "--mixup-alpha", str(MIXUP_ALPHA),
        "--cutmix-alpha", str(CUTMIX_ALPHA),
    ]
    if SMOKE:
        cmd.append("--no-save-every-epoch")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path

output_root = Path("/kaggle/working/plant_training_outputs")
for path in sorted(output_root.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_root))

## Deep analysis (no retraining)

Rebuild the deterministic test split, rerun each `best_model.pth`, and write
per-class / crop-vs-genus / **lab-vs-field** / confusion metrics plus top-k
accuracy. This is the linchpin for the generalization question: the
`source_analysis.csv` lab->field accuracy gap. Point `MODEL_ROOT` at an attached
models dataset to analyze already-trained models without retraining.

In [ ]:
# --- Deep analysis: reconstruct the test split, rerun best_model.pth, write metrics ---
# Does NOT retrain. Produces per-class / group / source (lab vs field) / confusion
# artifacts plus top-k accuracy under /kaggle/working/analysis, which the zip cell
# below bundles.
#
# Each trained model is attached as its OWN Kaggle input dataset (read-only), and
# analyze_kaggle.py expects a single <root>/<model_name>/best_model.pth (+ history.csv)
# tree. So stage the weights from every attached dataset into that layout under
# /kaggle/working, then point --model-root there. The image dataset itself is
# auto-detected by find_dataset_dir (it recurses /kaggle/input).
import csv
import shutil
import subprocess
from pathlib import Path

# canonical model name -> attached Kaggle input dataset holding that model's weights.
# Each dataset holds one *.pth (weights) and optionally one *.csv (training history).
MODEL_SOURCES = {
    "custom_cnn":                  "/kaggle/input/datasets/thngbuduc/cnn-v1",
    "custom_cnn_v2":               "/kaggle/input/datasets/thngbuduc/cnn-v2",
    "resnet50_feature_extraction": "/kaggle/input/datasets/thngbuduc/resnet50-feature-extraction",
    "resnet50_fine_tuning":        "/kaggle/input/datasets/thngbuduc/resnet50-fine-tuning-pt",
}
ANALYZE_MODELS = list(MODEL_SOURCES)  # trim this list to analyze fewer models
ANALYSIS_DIR = "/kaggle/working/analysis"


def pick_weights(pths):
    """Prefer best_model.pth; never pick a per-epoch checkpoint if a real one exists.

    Plain sorted()[0] would return epoch_10.pth before best_model.pth in a dataset
    that happens to contain both.
    """
    best = [p for p in pths if p.name == "best_model.pth"]
    if best:
        return best[0]
    non_epoch = [p for p in pths if not p.name.startswith("epoch_")]
    return (sorted(non_epoch) or sorted(pths))[0]


def history_epochs(csv_path):
    """Row count of a history.csv, used to flag smoke-test weights."""
    try:
        with open(csv_path, newline="") as fh:
            return max(0, sum(1 for _ in csv.reader(fh)) - 1)
    except Exception:
        return None


missing, suspicious = [], []
staged = Path("/kaggle/working/model_root")
if staged.exists():
    shutil.rmtree(staged)  # stale weights from an earlier run would be analyzed again

for model_name in ANALYZE_MODELS:
    src = Path(MODEL_SOURCES[model_name])
    if not src.exists():
        print(f"!! {model_name:<28} path does not exist: {src}")
        missing.append(model_name)
        continue
    pths = sorted(src.rglob("*.pth"))
    if not pths:
        print(f"!! {model_name:<28} no .pth found under {src}")
        missing.append(model_name)
        continue

    weights = pick_weights(pths)
    dst = staged / model_name
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights, dst / "best_model.pth")

    note = ""
    csvs = sorted(src.rglob("*.csv"))
    if csvs:
        shutil.copy2(csvs[0], dst / "history.csv")
        n = history_epochs(dst / "history.csv")
        note = f" + {csvs[0].name}" + (f" ({n} epochs)" if n is not None else "")
        if n is not None and n <= 1:
            suspicious.append(model_name)
            note += "  <-- !! 1 epoch: smoke-test weights, not real results"
    else:
        note = "  (no history.csv)"
    print(f"{model_name:<28} <- {weights.name}{note}")

present = [m for m in ANALYZE_MODELS if (staged / m / "best_model.pth").exists()]
if not present:
    raise SystemExit("No weights staged -- check the MODEL_SOURCES paths against the attached datasets.")
if missing:
    print(f"\n!! MISSING {len(missing)}/{len(ANALYZE_MODELS)}: {', '.join(missing)}")
    print("   Attach those datasets (Add Input, right panel) and rerun this cell.")
if suspicious:
    print(f"\n!! SMOKE WEIGHTS: {', '.join(suspicious)} -- do NOT put these numbers in the report.")

cmd = [
    "python", "scripts/analyze_kaggle.py",
    "--model-root", str(staged),
    "--output-dir", ANALYSIS_DIR,
    "--models", *present,
]
print(f"\nAnalyzing {len(present)}/{len(ANALYZE_MODELS)} models:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd="/kaggle/working/Internship")

print("\nArtifacts under", ANALYSIS_DIR, ":")
for path in sorted(Path(ANALYSIS_DIR).rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(ANALYSIS_DIR))

comparison = Path(ANALYSIS_DIR) / "analysis_comparison.csv"
if comparison.exists():
    rows = list(csv.DictReader(open(comparison, newline="")))
    print(f"\nanalysis_comparison.csv has {len(rows)} model row(s):",
          ", ".join(r["model_name"] for r in rows))
    if len(rows) < len(ANALYZE_MODELS):
        print(f"!! Expected {len(ANALYZE_MODELS)} -- the report needs all of them.")


## Training curves

Overlay the validation curves and print a per-model convergence table.

In [ ]:
# --- Training curves: overlay validation loss/accuracy for every model with history ---
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/Internship")
from plant_classifier.analysis import save_convergence_comparison

# In analyze mode the histories come from the analysis dir, where analyze_kaggle.py
# mirrors each attached model's history.csv. Only prefer plant_training_outputs when
# this session actually trained -- otherwise a leftover dir from an earlier run would
# silently plot that model instead of the four we just analyzed.
train_root = Path("/kaggle/working/plant_training_outputs")
analysis_root = Path("/kaggle/working/analysis")

if MODE == "train" and any(train_root.glob("*/history.csv")):
    curve_root = train_root
else:
    curve_root = analysis_root

curve_models = tuple(
    p.name
    for p in sorted(curve_root.iterdir())
    if p.is_dir() and (p / "history.csv").exists()
) if curve_root.exists() else ()

if curve_models:
    print(f"Curves from {curve_root} for {len(curve_models)} model(s):", ", ".join(curve_models))
    table = save_convergence_comparison(
        curve_root, curve_models, curve_root / "training_curves.png"
    )
    print("Wrote", curve_root / "training_curves.png")
    print("Per-model convergence (best epoch, val acc, overfit gap):")
    print(table.to_string(index=False))
else:
    print("No history.csv found under", curve_root,
          "- run the training or analysis cell first.")


## Grad-CAM: lab versus field

Render the saliency panels behind the domain-shift claim. For each genus this
pairs a Leafsnap **lab** image the reference model classifies correctly with a
**field** image of the *same genus* it gets wrong, then runs Grad-CAM for every
model on that pair. Holding the genus fixed means the label is identical, so a
difference in where the model looks is attributable to capture conditions alone.

Only a few dozen images are processed, so this is quick even without a GPU.

In [ ]:
# --- Grad-CAM: does attention leave the leaf on field images? ---
# Reuses the weights staged by the analysis cell above, so run that first.
# Writes <ANALYSIS_DIR>/gradcam/{<genus>_panel.png, gradcam_pairs.csv,
# gradcam_summary.csv}; the zip cell picks them up automatically.
import subprocess
from pathlib import Path

# The model the pairs are chosen from. Use the WEAKEST model: it produces the
# clearest field failures, and every model is then shown on those same pairs.
REFERENCE_MODEL = "custom_cnn"
MAX_PAIRS = 6           # one panel per genus

staged = Path("/kaggle/working/model_root")
gradcam_models = [
    m for m in ANALYZE_MODELS if (staged / m / "best_model.pth").exists()
]
if not gradcam_models:
    raise SystemExit("No staged weights -- run the analysis cell above first.")
if REFERENCE_MODEL not in gradcam_models:
    REFERENCE_MODEL = gradcam_models[0]
    print(f"Reference model not staged; falling back to {REFERENCE_MODEL}")

cmd = [
    "python", "scripts/gradcam_kaggle.py",
    "--model-root", str(staged),
    "--output-dir", ANALYSIS_DIR,
    "--models", *gradcam_models,          # explicit: the default list omits custom_cnn_v2
    "--reference-model", REFERENCE_MODEL,
    "--max-pairs", str(MAX_PAIRS),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd="/kaggle/working/Internship")

# Show the panels inline so the run can be judged without downloading the zip.
from IPython.display import Image, display

for panel in sorted(Path(ANALYSIS_DIR, "gradcam").glob("*_panel.png")):
    print(panel.name)
    display(Image(filename=str(panel)))


## Download bundle

Zip the figures + metrics needed for the report into one archive.

In [ ]:
# --- Bundle the outputs needed to write up the research, ready to download ---
# Zips every report artifact (figures + CSV/JSON metrics) from the training and
# analysis output dirs into one archive in /kaggle/working. Per-epoch checkpoints
# are always excluded; best_model.pth is excluded unless BUNDLE_WEIGHTS = True.
import zipfile
from pathlib import Path

BUNDLE_WEIGHTS = False  # True -> also include best_model.pth for each model (large)

SOURCE_DIRS = [
    Path("/kaggle/working/plant_training_outputs"),  # training: summaries, curves, per-model figs
    Path("/kaggle/working/analysis"),                # analyze_kaggle.py: per-class / source / confusion
]
BUNDLE_PATH = Path("/kaggle/working/plant_research_bundle.zip")


def keep(path: Path) -> bool:
    name = path.name
    if name.startswith("epoch_") and name.endswith(".pth"):
        return False  # per-epoch checkpoints are never needed for the writeup
    if name == "best_model.pth" and not BUNDLE_WEIGHTS:
        return False
    return True


count = 0
with zipfile.ZipFile(BUNDLE_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root in SOURCE_DIRS:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            if path.is_file() and keep(path):
                zf.write(path, path.relative_to(root.parent))  # keep dir name in the zip
                count += 1

if count:
    size_mb = BUNDLE_PATH.stat().st_size / 1e6
    print(f"Wrote {BUNDLE_PATH} ({count} files, {size_mb:.1f} MB)")
    print("Download it from the Kaggle output panel (Data > /kaggle/working) on the right.")
else:
    print("Nothing to bundle - run training (and optionally analyze_kaggle.py) first.")
